# Step 2: Train Q-Ensemble IQL (TripleCritic)

**CMPE 260 — Group 6**

This notebook trains Q-ensemble IQL with 3 Q-networks on all 3 D4RL environments.

**Runtime:** ~20 min per environment on Colab T4 GPU (60 min total)

**Outputs:**
- `results/training_hopper-medium-v2_3Q.csv`
- `results/training_halfcheetah-medium-v2_3Q.csv`
- `results/training_walker2d-medium-v2_3Q.csv`
- Checkpoints in `tmp/checkpoint_*/`

In [ ]:
!git clone -b sp1ffygeek_check_3 https://github.com/shloakk/iql-robustness-analysis.git
%cd iql-robustness-analysis

!pip install -q jax jaxlib flax optax
!pip install -q mujoco "gymnasium[mujoco]" gym
!pip install -q h5py tqdm matplotlib numpy scipy
!pip install -q absl-py ml_collections tensorboardX tensorflow-probability
!pip install -q git+https://github.com/Farama-Foundation/d4rl@master

In [ ]:
import os, sys, csv
import numpy as np
from tqdm import tqdm

sys.path.insert(0, '.')
import wrappers, gym, d4rl
from iql import Learner
from iql.dataset_utils import D4RLDataset, split_into_trajectories
from evaluation import evaluate

def normalize(dataset):
    trajs = split_into_trajectories(
        dataset.observations, dataset.actions, dataset.rewards,
        dataset.masks, dataset.dones_float, dataset.next_observations)
    def compute_returns(traj):
        return sum(rew for _, _, rew, _, _, _ in traj)
    trajs.sort(key=compute_returns)
    dataset.rewards /= compute_returns(trajs[-1]) - compute_returns(trajs[0])
    dataset.rewards *= 1000.0

def make_env_and_dataset(env_name, seed):
    env = gym.make(env_name)
    env = wrappers.EpisodeMonitor(env)
    env = wrappers.SinglePrecision(env)
    env.seed(seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    dataset = D4RLDataset(env)
    normalize(dataset)
    return env, dataset

ENVIRONMENTS = ['hopper-medium-v2', 'halfcheetah-medium-v2', 'walker2d-medium-v2']
NUM_CRITICS = 3  # TripleCritic (Q-ensemble)
MAX_STEPS = 300_000
SEED = 42
CONFIG = {
    'actor_lr': 3e-4, 'value_lr': 3e-4, 'critic_lr': 3e-4,
    'hidden_dims': (256, 256), 'discount': 0.99,
    'expectile': 0.7, 'temperature': 3.0,
    'dropout_rate': None, 'tau': 0.005,
}

In [ ]:
os.makedirs('results', exist_ok=True)

for env_name in ENVIRONMENTS:
    print(f"\n{'='*60}")
    print(f"Training Q-Ensemble IQL (3Q) on {env_name}")
    print(f"{'='*60}")

    env, dataset = make_env_and_dataset(env_name, SEED)
    agent = Learner(
        SEED, env.observation_space.sample()[np.newaxis],
        env.action_space.sample()[np.newaxis],
        max_steps=MAX_STEPS, num_critics=NUM_CRITICS, **CONFIG)

    eval_results = []
    for i in tqdm(range(1, MAX_STEPS + 1), desc=env_name):
        batch = dataset.sample(256)
        agent.update(batch)

        if i % 50_000 == 0:
            stats = evaluate(agent, env, 10)
            eval_results.append({'step': i, 'raw_return': stats['return'],
                                 'normalized_score': stats['return']})
            print(f"  Step {i:>7,}: score={stats['return']:.2f}")

    outfile = f'results/training_{env_name}_{NUM_CRITICS}Q.csv'
    with open(outfile, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['step', 'raw_return', 'normalized_score'])
        w.writeheader()
        w.writerows(eval_results)
    print(f"  Saved: {outfile}")

    save_dir = f'tmp/{env_name}_{NUM_CRITICS}Q'
    agent.save(save_dir, MAX_STEPS)
    print(f"  Checkpoint: {save_dir}/checkpoint_{MAX_STEPS}/")

print("\n✅ All Q-ensemble training complete!")

In [ ]:
from google.colab import files
for env_name in ENVIRONMENTS:
    files.download(f'results/training_{env_name}_{NUM_CRITICS}Q.csv')